In [1]:
import argparse
import os

# annotate the following lines to use CPU for data processing
# gpus = [0]
# os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
# os.environ["CUDA_VISIBLE_DEVICES"] = ','.join(map(str, gpus))
import psutil
import numpy as np
import math
import glob
import random
import itertools
import datetime
import time
import datetime
import sys
import scipy.io

import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid

from torch.utils.data import DataLoader
from torch.autograd import Variable
from torchsummary import summary
import torch.autograd as autograd
from torchvision.models import vgg19

import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.nn.init as init

from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
from sklearn.decomposition import PCA

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch import nn
from torch import Tensor
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce

import matplotlib.pyplot as plt
from torch.backends import cudnn
cudnn.benchmark = False
cudnn.deterministic = True

from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score
import pandas as pd
from openpyxl import Workbook
from scipy.signal import resample
from Dataloader_Excel import read_excel_eeg, read_excel_nirs

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
def print_memory_usage(stage):
    process = psutil.Process()
    memory_info = process.memory_info()
    print(f"[{stage}] Memory usage: {memory_info.rss / 1024**2:.2f} MB")

In [3]:
# Convolution module
# use conv to capture local features, instead of postion embedding.
class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40):
        # self.patch_size = patch_size
        super().__init__()

        self.shallownet = nn.Sequential(
            nn.Conv2d(1, 40, (1, 25), (1, 1)), # input_size, output_size, kernel_size, stride
            nn.Conv2d(40, 40, (30, 1), (1, 1)), # don't forget to change the number of the electrodes
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.AvgPool2d((1, 75), (1, 15)), # pooling acts as slicing to obtain 'patch' along the time dimension as in ViT
            nn.Dropout(0.7),
        )

        self.projection = nn.Sequential(
            nn.Conv2d(40, emb_size, (1, 1), stride=(1, 1)),  # transpose, conv could enhance fiting ability slightly
            Rearrange('b e (h) (w) -> b (h w) e'),
        )

    def forward(self, x: Tensor) -> Tensor:
        b, _, _, _ = x.shape
        x = self.shallownet(x)
        x = self.projection(x)
        return x

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: Tensor, mask: Tensor = None) -> Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)  
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)

        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out

In [5]:
# Reisdual Connection
class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )


class GELU(nn.Module):
    def forward(self, input: Tensor) -> Tensor:
        return input*0.5*(1.0+torch.erf(input/math.sqrt(2.0)))

In [6]:
class TransformerEncoderBlock(nn.Sequential):
    def __init__(self,
                 emb_size,
                 num_heads=10,
                 drop_p=0.7,
                 forward_expansion=4,
                 forward_drop_p=0.7):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(
                    emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )
            ))

In [7]:
class TransformerEncoder(nn.Sequential):
    def __init__(self, depth, emb_size):
        super().__init__(*[TransformerEncoderBlock(emb_size) for _ in range(depth)])

In [8]:
class ClassificationHead(nn.Sequential):
    def __init__(self, emb_size, n_classes):
        super().__init__()
        
        # global average pooling
        self.clshead = nn.Sequential(
            Reduce('b n e -> b e', reduction='mean'),
            nn.LayerNorm(emb_size),
            nn.Linear(emb_size, n_classes)
        )
        self.fc = nn.Sequential(
            nn.Linear(2440, 256), # 2440
            nn.ELU(),
            nn.Dropout(0.7),
            nn.Linear(256, 32),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(32, 2) # (32, n_classes), projects features extracted by Transformer to probabilities
        )

    def forward(self, x):
        x = x.contiguous().view(x.size(0), -1)
        out = self.fc(x)
        return x, out

In [9]:
class Conformer(nn.Sequential):
    def __init__(self, emb_size=40, depth=5, n_classes=2, **kwargs):
        super().__init__(

            PatchEmbedding(emb_size),
            TransformerEncoder(depth, emb_size),
            ClassificationHead(emb_size, n_classes)
        )

In [10]:
class ExP():
    def __init__(self, nsub):
        super(ExP, self).__init__()
        self.batch_size = 15
        self.n_epochs = 200
        self.c_dim = 4
        self.lr = 0.0001
        self.b1 = 0.5
        self.b2 = 0.999
        self.dimension = (190, 50)
        self.nSub = nsub

        self.start_epoch = 0
        self.root = 'Data/' # dataset directory

        self.log_write = open("MA_Results/log_subject%d.txt" % self.nSub, "w")
        self.log_excel = 'MA_Results//Log.xlsx'

        self.Tensor = torch.FloatTensor
        self.LongTensor = torch.LongTensor

        self.criterion_l1 = torch.nn.L1Loss()
        self.criterion_l2 = torch.nn.MSELoss()
        self.criterion_cls = torch.nn.CrossEntropyLoss()

        self.model = Conformer()

    # Segmentation and Reconstruction (S&R) data augmentation
    def interaug(self, timg, label):
        aug_data = []
        aug_label = []
        for cls4aug in range(2): # num_classes, 0, 1
            cls_idx = np.where(label == cls4aug)
            tmp_data = timg[cls_idx]
            tmp_label = label[cls_idx]

            num_aug_samples = int(self.batch_size / 2)
            tmp_aug_data = np.zeros((int(self.batch_size / 2), 1, 30, 1000)) # don't forget to change the data shape
            for ri in range(int(self.batch_size / 2)):
                for rj in range(8): # 8 * 125 points in one trial
                    rand_idx = np.random.randint(0, tmp_data.shape[0], 8)
                    tmp_aug_data[ri, :, :, rj * 125:(rj + 1) * 125] = tmp_data[rand_idx[rj], :, :,
                                                                      rj * 125:(rj + 1) * 125]

            aug_data.append(tmp_aug_data)
            aug_label.append(tmp_label[:int(self.batch_size / 2)])
        aug_data = np.concatenate(aug_data)
        aug_label = np.concatenate(aug_label)
        assert len(aug_data) == len(aug_label), "The numbers of data and labels don't match!"
        aug_shuffle = np.random.permutation(min(len(aug_data), len(aug_label)))
        aug_data = aug_data[aug_shuffle, :, :]
        aug_label = aug_label[aug_shuffle]

        aug_data = torch.from_numpy(aug_data)
        aug_data = aug_data.float()
        aug_label = torch.from_numpy(aug_label)
        aug_label = aug_label.long()
        return aug_data, aug_label

    def get_source_data(self):
        # train data
        self.train_data = []
        self.train_label = []
        self.total_data, self.total_label = read_excel_eeg(self.nSub) # (60, 30, 7000), (60,)
        self.total_data = resample(self.total_data, num=1000, axis=-1) # downsample eeg to (60, 30, 1000)
        indices = np.random.permutation(int(np.shape(self.total_data)[0]))
        train_len = int(np.shape(self.total_data)[0] * 0.6) # 60/40 training/testing
        train_indices = indices[:train_len]
        test_indices = indices[train_len:]
        self.train_data, self.train_label = self.total_data[train_indices], self.total_label[train_indices]
        
        print('Train Data', np.shape(self.train_data), np.shape(self.train_label)) # (trial, channel, sample_points) (triaL, 1)
        self.train_data = np.expand_dims(self.train_data, axis=1)
        self.train_label = np.transpose(self.train_label)

        self.allData = self.train_data
        self.allLabel = self.train_label.squeeze()

        shuffle_num = np.random.permutation(len(self.allData))
        self.allData = self.allData[shuffle_num, :, :, :]
        self.allLabel = self.allLabel[shuffle_num]

        # test data
        self.test_data, self.test_label = self.total_data[test_indices], self.total_label[test_indices]
        
        print('Test Data', np.shape(self.test_data), np.shape(self.test_label))
        self.test_data = np.expand_dims(self.test_data, axis=1)

        self.test_label = np.transpose(self.test_label)
        print(np.shape(self.test_label))

        self.testData = self.test_data
        self.testLabel = self.test_label.squeeze()

        self.allData = self.allData.astype(np.float32)
        self.testData = self.testData.astype(np.float32)

        # standardize
        target_mean = np.mean(self.allData)
        target_std = np.std(self.allData)
        self.allData -= target_mean
        self.allData /= target_std
        self.testData -= target_mean
        self.testData /= target_std

        # data shape: (trial, conv channel, electrode channel, time samples)
        return self.allData, self.allLabel, self.testData, self.testLabel

    def train(self):        
        img, label, test_data, test_label = self.get_source_data() # img = allData
        
        img = torch.from_numpy(img)
        label = torch.from_numpy(label)

        dataset = torch.utils.data.TensorDataset(img, label)
        self.dataloader = torch.utils.data.DataLoader(dataset=dataset, batch_size=self.batch_size, shuffle=True)

        test_data = torch.from_numpy(test_data)
        test_label = torch.from_numpy(test_label)
        test_dataset = torch.utils.data.TensorDataset(test_data, test_label)
        self.test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=self.batch_size, shuffle=True)

        # Optimizers
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, betas=(self.b1, self.b2))

        test_data = test_data.type(self.Tensor)
        test_label = test_label.type(self.LongTensor)

        best_acc = 0
        averAcc = 0
        averPrecision = 0
        averRecall = 0
        averF1 = 0
        averKappa = 0
        num = 0
        Y_true = 0
        Y_pred = 0

        # Train the cnn model
        total_step = len(self.dataloader)
        curr_lr = self.lr

        #train_loss = []
        #test_loss = []
        
        acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []

        for e in range(self.n_epochs):
            # in_epoch = time.time()
            self.model.train()
            #epoch_train_loss = 0
            for i, (img, label) in enumerate(self.dataloader):
                print(f"Batch {i}: img shape {img.shape}, label shape {label.shape}")
       
                img = Variable(img.type(self.Tensor))
                label = Variable(label.type(self.LongTensor))

                # data augmentation
                aug_data, aug_label = self.interaug(self.allData, self.allLabel)
                img = torch.cat((img, aug_data))
                label = torch.cat((label, aug_label))

                tok, outputs = self.model(img)

                loss = self.criterion_cls(outputs, label) 
                #epoch_train_loss += loss
                
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
            # out_epoch = time.time()
            #train_loss.append(epoch_train_loss / len(self.dataloader))

            # test process
            if (e + 1) % 1 == 0:
                self.model.eval()
                Tok, Cls = self.model(test_data)
                loss_test = self.criterion_cls(Cls, test_label)            

                y_pred = torch.max(Cls, 1)[1].cpu().numpy()
                y_true = test_label.cpu().numpy()

                precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
                kappa = cohen_kappa_score(y_true, y_pred)

                train_pred = torch.max(outputs, 1)[1].cpu().numpy()
                train_true = label.cpu().numpy()
                train_acc = (train_pred == train_true).sum() / train_true.shape[0]
                acc = (y_pred == y_true).sum() / y_true.shape[0]

                print(f"Epoch: {e}  Train loss: {loss.detach().cpu().numpy():.2f}  Test loss: {loss_test.detach().cpu().numpy():.2f}")
                print(f"Train accuracy: {train_acc:.2f}  Test accuracy: {acc:.2f}")
                print(f"Precision: {precision:.2f}  Recall: {recall:.2f}  F1-score: {f1:.2f}  Kappa: {kappa:.2f}")

                self.log_write.write(str(e) + "    " + str(acc) + "\n")
                self.log_write.write(f"Precision: {precision:.2f}  Recall: {recall:.2f}  F1-score: {f1:.2f}  Kappa: {kappa:.2f}\n")

                num += 1
                acc_list.append(acc)
                if len(acc_list) <= 1:
                    best_acc = acc
                else:
                    best_acc = max(acc_list[:-1])
                precision_list.append(precision)
                recall_list.append(recall)
                f1_list.append(f1)
                kappa_list.append(kappa)
                if acc > best_acc:
                    best_acc = acc
                    Y_true = test_label
                    Y_pred = y_pred

        torch.save(self.model.state_dict(), 'model.pth') # saving the model after the last epoch
        
        mean_acc, std_acc = np.mean(acc_list[-50:]), np.std(acc_list[-50:])
        best_acc = max(acc_list)
        mean_prec, std_prec = np.mean(precision_list[-50:]), np.std(precision_list[-50:])
        mean_rec, std_rec = np.mean(recall_list[-50:]), np.std(recall_list[-50:])
        mean_f1, std_f1 = np.mean(f1_list[-50:]), np.std(f1_list[-50:])
        mean_kappa = np.mean(kappa_list[-50:])

        mean_acc, std_acc = mean_acc * 100, std_acc * 100
        mean_prec, std_prec = mean_prec * 100, std_prec * 100
        mean_rec, std_rec = mean_rec * 100, std_rec * 100
        mean_f1, std_f1 = mean_f1 * 100, std_f1 * 100

        print(f"Accuracy: {mean_acc:.2f} ± {std_acc:.2f}")
        print(f"Precision: {mean_prec:.2f} ± {std_prec:.2f}")
        print(f"Recall: {mean_rec:.2f} ± {std_rec:.2f}")
        print(f"F1-score: {mean_f1:.2f} ± {std_f1:.2f}")
        print(f"Kappa: {mean_kappa:.2f}")

        self.log_write.write(f"Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n")
        self.log_write.write(f"Precision: {mean_prec:.2f} ± {std_prec:.2f}\n")
        self.log_write.write(f"Recall: {mean_rec:.2f} ± {std_rec:.2f}\n")
        self.log_write.write(f"F1-score: {mean_f1:.2f} ± {std_f1:.2f}\n")
        self.log_write.write(f"Kappa: {mean_kappa:.2f}\n")

        new_row = pd.DataFrame([[mean_acc, std_acc, best_acc, mean_prec, std_prec, mean_rec, std_rec, mean_f1, std_f1, mean_kappa]], 
                           columns=['Accuracy', 'Std_Accuracy', 'Best_Accuracy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
        new_row = new_row.round(2)
        if not os.path.exists(self.log_excel):
            wb = Workbook()
            ws = wb.active
            ws.title = 'Sheet1'
            ws.append(['Accuracy', 'Std_Accuracy', 'Best_Accuracy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
            wb.save(self.log_excel)
        with pd.ExcelWriter(self.log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
            new_row.to_excel(writer, index=False, header=False, startrow=writer.sheets['Sheet1'].max_row)

        return best_acc, mean_acc, Y_true, Y_pred

def main():
    best = 0
    aver = 0
    result_path = "MA_Results/sub_result.txt"
    
    if not os.path.exists('MA_Results'):
        os.makedirs('MA_Results')

    # 29 subjects
    with open(result_path, "w") as result_write:
        for i in range(29):
            starttime = datetime.datetime.now()
    
            # random seed
            seed_n = np.random.randint(2021)
            print('seed is ' + str(seed_n))
            random.seed(seed_n)
            np.random.seed(seed_n)
            torch.manual_seed(seed_n)
    
            print('Subject %d' % (i+1))
            exp = ExP(i + 1)
    
            best_acc, averAcc, Y_true, Y_pred = exp.train() # start training
            print('THE BEST ACCURACY IS ' + str(best_acc))
    
            # write results to the log file
            result_write.write(f'Subject {i + 1} : Seed is: {seed_n}\n')
            result_write.write(f'Subject {i + 1} : The best accuracy is: {best_acc}\n')
            result_write.write(f'Subject {i + 1} : The average accuracy is: {averAcc}\n')
    
            endtime = datetime.datetime.now()
            print(f'Subject {i+1} duration: {endtime - starttime}')
    
            # accumulated best and average accuracies
            best += best_acc
            aver += averAcc
  
    best /= 29 # number of subjects
    aver /= 29
    with open(result_path, "w") as result_write:
        result_write.write(f'**The average Best accuracy is: {best}\n')
        result_write.write(f'The average Aver accuracy is: {aver}\n')
    
    print("Results saved to:", result_path)

if __name__ == "__main__":
    print(time.asctime(time.localtime(time.time())))
    main()
    print(time.asctime(time.localtime(time.time())))

Mon Apr 14 20:12:43 2025
seed is 1131
Subject 1
1 EEG (60, 30, 4000) Labels (60,)
Train Data (36, 30, 1000) (36,)
Test Data (24, 30, 1000) (24,)
(24,)
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 0  Train loss: 0.80  Test loss: 0.69
Train accuracy: 0.45  Test accuracy: 0.58
Precision: 0.34  Recall: 0.58  F1-score: 0.43  Kappa: 0.00
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])
Epoch: 1  Train loss: 0.65  Test loss: 0.68
Train accuracy: 0.55  Test accuracy: 0.62
Precision: 0.77  Recall: 0.62  F1-score: 0.52  Kappa: 0.11
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])
Epoch: 2  Train loss: 0.80  Test loss: 0.68
Train accuracy: 0.40  Test accuracy: 0.79
Precision: 0.79  Recall: 0.79  F1-score: 0.79  Kappa: 0.57
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img sha

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 0  Train loss: 0.77  Test loss: 0.73
Train accuracy: 0.55  Test accuracy: 0.50
Precision: 0.25  Recall: 0.50  F1-score: 0.33  Kappa: 0.00
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])


...

Epoch: 0  Train loss: 0.80  Test loss: 0.69
Train accuracy: 0.50  Test accuracy: 0.50
Precision: 0.25  Recall: 0.50  F1-score: 0.33  Kappa: 0.00
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 1  Train loss: 0.71  Test loss: 0.66
Train accuracy: 0.55  Test accuracy: 0.50
Precision: 0.25  Recall: 0.50  F1-score: 0.33  Kappa: 0.00
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])
Epoch: 2  Train loss: 0.74  Test loss: 0.64
Train accuracy: 0.45  Test accuracy: 0.83
Precision: 0.88  Recall: 0.83  F1-score: 0.83  Kappa: 0.67
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])
Epoch: 3  Train loss: 0.79  Test loss: 0.62
Train accuracy: 0.45  Test accuracy: 0.79
Precision: 0.79  Recall: 0.79  F1-score: 0.79  Kappa: 0.58
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img sha

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 0  Train loss: 0.92  Test loss: 0.66
Train accuracy: 0.25  Test accuracy: 0.62
Precision: 0.39  Recall: 0.62  F1-score: 0.48  Kappa: 0.00
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])
Epoch: 1  Train loss: 0.81  Test loss: 0.67
Train accuracy: 0.35  Test accuracy: 0.54
Precision: 0.37  Recall: 0.54  F1-score: 0.44  Kappa: -0.16
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 2: img shape torch.Size([6, 1, 30, 1000]), label shape torch.Size([6])
Epoch: 2  Train loss: 0.84  Test loss: 0.67
Train accuracy: 0.35  Test accuracy: 0.54
Precision: 0.37  Recall: 0.54  F1-score: 0.44  Kappa: -0.16
Batch 0: img shape torch.Size([15, 1, 30, 1000]), label shape torch.Size([15])
Batch 1: img s